---
## 4. Construção da Base Final Analítica

Unimos todas as tabelas em uma base única que permite responder às três frentes de análise:

```
orders_clean  ←→  customers     (customer_id)
              ←→  order_items   (order_id)  → agrega preço, frete, seller
              ←→  reviews       (order_id)  → nota do cliente
              ←→  products      (product_id) → categoria do produto
              ←→  sellers       (seller_id)  → estado do vendedor
```

In [ ]:
import pandas as pd


In [11]:
df_olist_customers = pd.read_csv("../olist_customers_dataset.csv")
df_olist_geolocation = pd.read_csv("../olist_geolocation_dataset.csv")
olist_order_items = pd.read_csv("../olist_order_items_dataset.csv")
olist_order_payments = pd.read_csv("../olist_order_payments_dataset.csv")
olist_order_reviews = pd.read_csv("../olist_order_reviews_dataset.csv")
olist_orders = pd.read_csv("../olist_orders_dataset.csv")
olist_products = pd.read_csv("../olist_products_dataset.csv")
olist_sellers = pd.read_csv("../olist_sellers_dataset.csv")
df_orders_delivered = pd.read_pickle("../df_orders_delivered.pkl")
df_reviews_clean = pd.read_pickle("../df_reviews_clean.pkl")
df_orders_clean = pd.read_pickle("../df_orders_clean.pkl")
df_items_agg = pd.read_pickle("../df_items_agg.pkl")
df_products_clean = pd.read_pickle("../df_products_clean.pkl")


In [12]:
# ============================================================
# CONSTRUÇÃO DA BASE FINAL ANALÍTICA
# ============================================================
# Realizamos os joins sequenciais usando LEFT JOIN em todos
# para preservar todos os pedidos, mesmo sem correspondência
# nas tabelas auxiliares (ex: pedido sem avaliação).
# ============================================================

df_base = (
    df_orders_clean

    # Adiciona informações do cliente: estado e cidade
    .merge(
        df_olist_customers[['customer_id','customer_unique_id','customer_state','customer_city']],
        on='customer_id',
        how='left'
    )

    # Adiciona métricas financeiras e o vendedor/produto principal do pedido
    .merge(df_items_agg, on='order_id', how='left')

    # Adiciona a nota de avaliação do cliente
    .merge(df_reviews_clean, on='order_id', how='left')

    # Adiciona a categoria do produto (em inglês)
    .merge(
        df_products_clean[['product_id','product_category_name_english']],
        on='product_id',
        how='left'
    )

    # Adiciona o estado do vendedor
    .merge(
        olist_sellers[['seller_id','seller_state','seller_city']],
        on='seller_id',
        how='left'
    )
)

print(f'✅ Base final construída: {df_base.shape[0]:,} pedidos x {df_base.shape[1]} colunas')
print('\nVerificação de nulos na base final:')
for col in df_base.columns:
    nulls = df_base[col].isnull().sum()
    if nulls > 0:
        pct = nulls / len(df_base) * 100
        print(f'  ⚠️  {col}: {nulls:,} nulos ({pct:.1f}%)')
    else:
        print(f'  ✅  {col}')

✅ Base final construída: 96,456 pedidos x 25 colunas

Verificação de nulos na base final:
  ✅  order_id
  ✅  customer_id
  ✅  order_status
  ✅  order_purchase_timestamp
  ⚠️  order_approved_at: 14 nulos (0.0%)
  ⚠️  order_delivered_carrier_date: 1 nulos (0.0%)
  ✅  order_delivered_customer_date
  ✅  order_estimated_delivery_date
  ⚠️  tempo_aprovacao_h: 14 nulos (0.0%)
  ✅  tempo_entrega_dias
  ✅  tempo_estimado_dias
  ✅  atraso_dias
  ✅  atrasado
  ✅  customer_unique_id
  ✅  customer_state
  ✅  customer_city
  ✅  total_price
  ✅  total_freight
  ✅  n_items
  ✅  seller_id
  ✅  product_id
  ⚠️  review_score: 645 nulos (0.7%)
  ✅  product_category_name_english
  ✅  seller_state
  ✅  seller_city


In [13]:
# ============================================================
# COLUNAS DERIVADAS PARA ANÁLISE
# ============================================================
# Criamos colunas calculadas que facilitam as análises
# e a apresentação dos resultados de forma executiva.
# ============================================================

# --- FAIXA DE TEMPO DE ENTREGA ---
# Agrupa o tempo de entrega em faixas para análise por bucket
# Permite ver em qual faixa a satisfação começa a cair
def faixa_entrega(dias):
    if dias <= 7:    return '1. Até 7 dias'
    elif dias <= 14: return '2. 8–14 dias'
    elif dias <= 21: return '3. 15–21 dias'
    elif dias <= 30: return '4. 22–30 dias'
    else:            return '5. Mais de 30 dias'

df_base['faixa_entrega'] = df_base['tempo_entrega_dias'].apply(faixa_entrega)

# --- CLASSIFICAÇÃO DA AVALIAÇÃO ---
# Simplifica o score de 1-5 em 3 categorias executivas
# Facilita a comunicação com stakeholders não técnicos
def classifica_review(score):
    if score >= 4:   return 'Positiva (4–5)'
    elif score == 3: return 'Neutra (3)'
    else:            return 'Negativa (1–2)'

df_base['classificacao_review'] = df_base['review_score'].apply(
    lambda x: classifica_review(x) if pd.notna(x) else 'Sem avaliação'
)

# --- REGIÃO DO CLIENTE ---
# Agrupa os 27 estados em 5 regiões para análise macro
# Permite identificar padrões regionais de SLA e satisfação
regioes = {
    'AC':'Norte', 'AM':'Norte', 'AP':'Norte', 'PA':'Norte',
    'RO':'Norte', 'RR':'Norte', 'TO':'Norte',
    'AL':'Nordeste', 'BA':'Nordeste', 'CE':'Nordeste', 'MA':'Nordeste',
    'PB':'Nordeste', 'PE':'Nordeste', 'PI':'Nordeste', 'RN':'Nordeste', 'SE':'Nordeste',
    'DF':'Centro-Oeste', 'GO':'Centro-Oeste', 'MS':'Centro-Oeste', 'MT':'Centro-Oeste',
    'ES':'Sudeste', 'MG':'Sudeste', 'RJ':'Sudeste', 'SP':'Sudeste',
    'PR':'Sul', 'RS':'Sul', 'SC':'Sul'
}
df_base['regiao_cliente'] = df_base['customer_state'].map(regioes)

# --- TICKET TOTAL ---
# Soma do valor dos produtos + frete por pedido
df_base['ticket_total'] = df_base['total_price'] + df_base['total_freight']

print('✅ Colunas derivadas criadas com sucesso!')
print(f'\nFaixa de entrega (distribuição):')
print(df_base['faixa_entrega'].value_counts().sort_index().to_string())
print(f'\nClassificação de reviews:')
print(df_base['classificacao_review'].value_counts().to_string())
print(f'\nRegião do cliente:')
print(df_base['regiao_cliente'].value_counts().to_string())

✅ Colunas derivadas criadas com sucesso!

Faixa de entrega (distribuição):
faixa_entrega
1. Até 7 dias         33696
2. 8–14 dias          36397
3. 15–21 dias         15369
4. 22–30 dias          6891
5. Mais de 30 dias     4103

Classificação de reviews:
classificacao_review
Positiva (4–5)    75631
Negativa (1–2)    12266
Neutra (3)         7914
Sem avaliação       645

Região do cliente:
regiao_cliente
Sudeste         66188
Sul             13811
Nordeste         9040
Centro-Oeste     5623
Norte            1794


In [14]:
df_base.to_csv("../base_analitica_final.csv", index=False)